# 2 Preprocessing Data

Read in the Google Trends keyword interest data for preprocessing.

In [1]:
from pathlib import Path
import pandas as pd

In [2]:
DATA_DIR = Path.cwd()
TRENDS_PATH = DATA_DIR / "trends_final_keywords.csv"

trends = pd.read_csv(TRENDS_PATH, parse_dates=["date"])
trends.head()

,date,keyword,interest
0,2016-05-01,home care,100.0
1,2016-06-01,home care,60.0
2,2016-07-01,home care,90.0
3,2016-08-01,home care,100.0
4,2016-09-01,home care,90.0


In [4]:
rows_before = len(trends)
nan_rows = trends.isna().any(axis=1)
keywords_with_nan = trends.loc[nan_rows, "keyword"].dropna().nunique()

trends = trends.dropna().reset_index(drop=True)
rows_dropped = rows_before - len(trends)

print(f"Keywords with at least one NaN row: {keywords_with_nan:,}")
print(f"Rows dropped: {rows_dropped:,}")

Keywords with at least one NaN row: 0
Rows dropped: 0


In [6]:
trends.info()
trends[["keyword", "interest"]].describe(include="all")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 174966 entries, 0 to 174965
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype         
---  ------    --------------   -----         
 0   date      174966 non-null  datetime64[ns]
 1   keyword   174966 non-null  object        
 2   interest  174966 non-null  float64       
dtypes: datetime64[ns](1), float64(1), object(1)
memory usage: 4.0+ MB


,keyword,interest
count,174966,174966.000000
unique,1446,NaN
top,home care,NaN
freq,121,NaN
mean,NaN,21.387612
std,NaN,153.676989
min,NaN,0.000000
25%,NaN,0.000000
50%,NaN,0.000000
75%,NaN,5.263158


In [7]:
zero_threshold = 0.70

zero_share_by_keyword = trends.groupby("keyword")["interest"].apply(lambda x: (x == 0).mean())
keywords_to_drop = zero_share_by_keyword[zero_share_by_keyword >= zero_threshold].index
drop_zero_heavy_rows = trends["keyword"].isin(keywords_to_drop)
rows_to_drop = drop_zero_heavy_rows.sum()

print(f"Keywords with at least {zero_threshold:.0%} zeros: {len(keywords_to_drop):,}")
print(f"Rows dropped: {rows_to_drop:,}")

trends = trends.loc[~drop_zero_heavy_rows].reset_index(drop=True)

print(f"Rows remaining: {len(trends):,}")

zero_share_by_keyword.loc[keywords_to_drop].sort_values(ascending=False).head(10)

Keywords with at least 70% zeros: 967
Rows dropped: 117,007
Rows remaining: 57,959


keyword
internet retail             1.0
lng exports                 1.0
internet infrastructure     1.0
5g spectrum                 1.0
subscription economy        1.0
student loan refinancing    1.0
stress tests                1.0
ism manufacturing           1.0
streaming subscribers       1.0
kellogg stock               1.0
Name: interest, dtype: float64

In [8]:
# Check how many keywords/rows would be dropped at different mean-interest thresholds

mean_interest_by_keyword = trends.groupby("keyword")["interest"].mean()

thresholds = [0.5, 1, 2, 5, 10, 20, 50]

mean_filter_summary = []

for threshold in thresholds:
    keywords_to_drop = mean_interest_by_keyword[mean_interest_by_keyword < threshold].index
    rows_to_drop = trends["keyword"].isin(keywords_to_drop).sum()
    
    mean_filter_summary.append({
        "mean_interest_threshold": threshold,
        "keywords_dropped": len(keywords_to_drop),
        "rows_dropped": rows_to_drop,
        "keywords_remaining": trends["keyword"].nunique() - len(keywords_to_drop),
        "rows_remaining": len(trends) - rows_to_drop,
    })

mean_filter_summary = pd.DataFrame(mean_filter_summary)
mean_filter_summary


,mean_interest_threshold,keywords_dropped,rows_dropped,keywords_remaining,rows_remaining
0,0.5,0,0,479,57959
1,1.0,0,0,479,57959
2,2.0,14,1694,465,56265
3,5.0,91,11011,388,46948
4,10.0,234,28314,245,29645
5,20.0,319,38599,160,19360
6,50.0,394,47674,85,10285
